# Par3 Literary Translation Generation

Generates LLM translations for a filtered, cross-language pool of Par3 paragraphs (Karpinska et al., 2022)

**Data basis:**
- Paragraphs pooled from German, Russian and Chinese source books where >= `MIN_TRANSLATORS` (3) different human translators simultaneously render the same passage at >= `MIN_SENTENCES` (10) sentences
- 891 paragraphs pooled (92 German, 633 Russian, 166 Chinese), stratified down to `N_PARAGRAPHS` (200) drawn as evenly as possible across the three languages (67/67/66) so the much larger Russian pool doesn't swamp German
- See `data_description/descriptive_analysis_par3.ipynb` for the full language comparison behind this choice

**Models**

| Key | Model |
|---|---|
| `llama` | meta-llama/Llama-3.1-8B-Instruct |
| `mistral` | mistralai/Mistral-7B-Instruct-v0.3 |
| `qwen` | Qwen/Qwen2.5-7B-Instruct |
| `gemma` | google/gemma-2-9b-it |

All four are open-weight models run locally via vLLM (identical setup to the WritingPrompts/SummEval/Multi-News generation notebooks)

**Setup**

- GPU: an A100 is recommended over an L4 here
- Run one model per session
- vLLM does not release GPU memory reliably, so after each model restart the runtime and change `MODEL_KEY`
- Requires `par3.pkl` uploaded to Drive at `<Masterarbeit>/data/par3/par3.pkl` (same relative layout as the local repo's `data/par3/par3.pkl`)

**Decoding configuration**

Fixed across all models, identical to the other three tasks:
- `temperature=1.0`
- `top_p=0.9`
- `repetition_penalty=1.0` (disabled)
- `n=5`

No length instruction in the prompt

## Installation

In [ ]:
# Restart the runtime after this cell
!pip install -q -U vllm nltk
!pip install -q --upgrade huggingface_hub
!pip uninstall -y -q torchaudio torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.

In [1]:
import json
import pathlib
import random
import re
import time
from google.colab import drive


# Mount Drive
drive.mount('/content/drive')
OUT_DIR = pathlib.Path('/content/drive/MyDrive/Masterarbeit/generations')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT_DIR)

Mounted at /content/drive
Output directory: /content/drive/MyDrive/Masterarbeit/generations


## Configuration

In [2]:
# Switch this between runs: 'llama' | 'mistral' | 'qwen' | 'gemma'
MODEL_KEY = 'gemma'


MODELS = {
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
    'gemma':   'google/gemma-2-9b-it',
}

MODEL_ID = MODELS[MODEL_KEY]

# Sampling
N_PARAGRAPHS = 200  # paragraphs drawn from the pool, stratified evenly across source languages
N_SAMPLES = 5        # generations per paragraph (n=5)
SEED = 42            # controls both paragraph sampling and generation

# Pooling filter (see data_description/descriptive_analysis_par3.ipynb for the
# analysis behind these numbers):
# paragraphs where at least MIN_TRANSLATORS reach MIN_SENTENCES sentences,
# pooled across German + Russian + Chinese source books
# -> yields 891 paragraphs (92 DE + 633 RU + 166 ZH), stratified down to
#    N_PARAGRAPHS below

MIN_SENTENCES = 10
MIN_TRANSLATORS = 3
POOL_LANGUAGES = {'de': 'German', 'ru': 'Russian', 'zh': 'Chinese'}

# Decoding (identical to WritingPrompts/SummEval/Multi-News)
TEMPERATURE = 1.0
TOP_P = 0.9
REPETITION_PENALTY = 1.0  # deliberately disabled

# Token budget: paragraphs that clear MIN_SENTENCES range hugely in length
# (median ~450 words in the human translations, but up to ~5500 in the
# longest single one)

SRC_MAX_TOKENS = 3072
MAX_NEW_TOKENS = 3072
MAX_MODEL_LEN = 8192  # SRC_MAX_TOKENS + MAX_NEW_TOKENS + template/system overhead + margin

random.seed(SEED)

print(f'Model:  {MODEL_ID}')
print(f'Pooled languages: {list(POOL_LANGUAGES.values())}')

Model:  google/gemma-2-9b-it
Pooled languages: ['German', 'Russian', 'Chinese']


In [3]:
# Llama and Gemma are gated (accept the license on each model page and store
# token in a .env file on Drive):
#   /content/drive/MyDrive/Masterarbeit/.env
# containing a single line:
#   HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxx

if MODEL_KEY in ('llama', 'gemma'):
    from huggingface_hub import login

    env_path = OUT_DIR.parent / '.env'

    # Read Hugging Face token from .env file on Drive
    hf_token = dict(
        line.strip().split('=', 1)
        for line in env_path.read_text().splitlines()
        if line.strip()
    )['HF_TOKEN']

    login(token=hf_token)
    print('Logged in to Hugging Face')

Logged in to Hugging Face


## Load & filter Par3

- Loaded from a local pickle (not a HuggingFace dataset like the other three tasks) -> `par3.pkl` must already be on Drive
- Only books with >= `MIN_TRANSLATORS` translator slots can possibly qualify at all
- Within those, only paragraph indices where that many translators simultaneously reach >= `MIN_SENTENCES` sentences are pooled
- `human_translations` keeps only the *qualifying* translators for that index (>= `MIN_SENTENCES`)

In [4]:
import pickle
import os

# The Colab egress proxy blocks nltk.download's raw.githubusercontent.com fetch
os.environ.setdefault('NLTK_ALLOW_PROXIED_URLOPEN', '1')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

PAR3_PATH = OUT_DIR.parent / 'data' / 'par3.pkl'

with open(PAR3_PATH, 'rb') as f:
    par3 = pickle.load(f)

print(f'Loaded {len(par3)} books from {PAR3_PATH}')


def n_sentences(text):
    if not text or not text.strip():
        return 0
    return len(sent_tokenize(text))


# Pool paragraphs across the three eligible languages
pooled = []
for lang_suffix, lang_name in POOL_LANGUAGES.items():
    books = {k: v for k, v in par3.items() if k.endswith(f'_{lang_suffix}')}
    for book_id, book in books.items():
        translators = list(book['translator_data'].items())
        if len(translators) < MIN_TRANSLATORS:
            continue
        for i in range(len(book['source_paras'])):
            qualifying = {
                tname: tdata['translator_paras'][i]
                for tname, tdata in translators
                if i < len(tdata['translator_paras'])
                and n_sentences(tdata['translator_paras'][i]) >= MIN_SENTENCES
            }
            if len(qualifying) < MIN_TRANSLATORS:
                continue
            pooled.append({
                'book_id': book_id,
                'source_lang': lang_name,
                'paragraph_index': i,
                'source_text': book['source_paras'][i],
                'gt_translation': book['gt_paras'][i] if i < len(book['gt_paras']) else None,
                'human_translations': qualifying,
                'n_qualifying_translators': len(qualifying),
            })

print(f'{len(pooled)} pooled paragraphs (>= {MIN_TRANSLATORS} translators, >= {MIN_SENTENCES} sentences)')

from collections import Counter
print('By language:', dict(Counter(p['source_lang'] for p in pooled)))
print('By book:', dict(Counter(p['book_id'] for p in pooled)))

Loaded 113 books from /content/drive/MyDrive/Masterarbeit/data/par3.pkl
891 pooled paragraphs (>= 3 translators, >= 10 sentences)
By language: {'German': 92, 'Russian': 633, 'Chinese': 166}
By book: {'the_metamorphosis_de': 15, 'heidi_de': 4, 'the_notebooks_of_malte_laurids_brigge_de': 29, 'the_trial_de': 32, 'death_in_venice_de': 12, 'the_death_of_ivan_ilyich_ru': 10, 'dead_souls_ru': 59, 'crime_and_punishment_ru': 130, 'petersburg_ru': 1, 'oblomov_ru': 4, 'notes_from_underground_ru': 25, 'fathers_and_sons_ru': 12, 'brothers_karamazov_ru': 201, 'the_gentleman_from_san_francisco_ru': 1, 'the_idiot_ru': 134, 'we_ru': 10, 'poor_folk_ru': 46, 'the_journey_to_the_west_zh': 166}


In [5]:
# Stratify down to N_PARAGRAPHS: split N_PARAGRAPHS as evenly as possible across the three source languages

from collections import defaultdict

pooled_all = pooled

by_lang = defaultdict(list)
for item in pooled_all:
    by_lang[item['source_lang']].append(item)

langs = sorted(by_lang)
base_quota, remainder = divmod(N_PARAGRAPHS, len(langs))
quotas = {lang: base_quota + (1 if i < remainder else 0) for i, lang in enumerate(langs)}

sampler = random.Random(SEED)
pooled = [item for lang in langs for item in sampler.sample(by_lang[lang], quotas[lang])]
sampler.shuffle(pooled)

print(f'Sampled {len(pooled)} of {len(pooled_all)} pooled paragraphs (quota: {quotas})')
print('By language:', dict(Counter(p['source_lang'] for p in pooled)))
print('By book:', dict(Counter(p['book_id'] for p in pooled)))

Sampled 200 of 891 pooled paragraphs (quota: {'Chinese': 67, 'German': 67, 'Russian': 66})
By language: {'Russian': 66, 'German': 67, 'Chinese': 67}
By book: {'notes_from_underground_ru': 2, 'the_trial_de': 25, 'the_notebooks_of_malte_laurids_brigge_de': 22, 'the_journey_to_the_west_zh': 67, 'the_metamorphosis_de': 9, 'dead_souls_ru': 4, 'the_idiot_ru': 19, 'brothers_karamazov_ru': 19, 'crime_and_punishment_ru': 14, 'poor_folk_ru': 4, 'death_in_venice_de': 9, 'heidi_de': 2, 'oblomov_ru': 1, 'we_ru': 1, 'the_death_of_ivan_ilyich_ru': 2}


In [6]:
# Assign prompt_ids and save the pooled paragraphs to a JSONL file for later use
for i, item in enumerate(pooled):
    item['prompt_id'] = f'par3_{i:04d}'

with open(OUT_DIR / 'par3_paragraphs_sample.jsonl', 'w') as f:
    for item in pooled:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Saved {len(pooled)} paragraphs')
print()
print('Example:')
print(f"{pooled[0]['prompt_id']}  ({pooled[0]['source_lang']}, {pooled[0]['book_id']}, "
      f"{pooled[0]['n_qualifying_translators']} qualifying translators)")
print(pooled[0]['source_text'][:300])

Saved 200 paragraphs

Example:
par3_0000  (Russian, notes_from_underground_ru, 3 qualifying translators)
Впрочем, что ж я? — все это делают; болезнями-то и тщеславятся, а я, пожалуй, и больше всех. Не будем спорить; мое возражение нелепо. Но все-таки я крепко убежден, что не только очень много сознания, но даже и всякое сознание болезнь. Я стою на том. Оставим и это на минуту. Скажите мне вот что: отче


## Length distribution & token budget

Checking the `SRC_MAX_TOKENS`/`MAX_NEW_TOKENS` choice from Configuration against
the actual pooled data before spending GPU time on it

In [7]:
import numpy as np
from nltk.tokenize import word_tokenize

src_words = np.array([len(word_tokenize(p['source_text'])) for p in pooled])
human_words = np.array([
    len(word_tokenize(text))
    for p in pooled for text in p['human_translations'].values()
])

print('SOURCE TEXT word counts:')
print(f'  mean={src_words.mean():.0f} median={np.median(src_words):.0f} '
      f'min={src_words.min()} max={src_words.max()}')
print('HUMAN TRANSLATION word counts (all qualifying translators):')
print(f'  mean={human_words.mean():.0f} median={np.median(human_words):.0f} '
      f'min={human_words.min()} max={human_words.max()} '
      f'p95={np.percentile(human_words, 95):.0f} p99={np.percentile(human_words, 99):.0f}')

TOKENS_PER_WORD = 1.5  # generous upper bound; the live tokenizer is the source of truth
src_over_budget = np.mean(src_words * TOKENS_PER_WORD > SRC_MAX_TOKENS)
human_over_budget = np.mean(human_words * TOKENS_PER_WORD > MAX_NEW_TOKENS)

print()
print(f'~{100*src_over_budget:.1f}% of source paragraphs exceed the {SRC_MAX_TOKENS}-token '
      f'source budget (will be truncated, see truncate_to_tokens below)')
print(f'~{100*human_over_budget:.1f}% of human translations exceed the {MAX_NEW_TOKENS}-token '
      f'generation budget (a model matching that length would get cut off)')

SOURCE TEXT word counts:
  mean=325 median=242 min=1 max=4943
HUMAN TRANSLATION word counts (all qualifying translators):
  mean=520 median=372 min=84 max=5529 p95=1330 p99=3570

~1.5% of source paragraphs exceed the 3072-token source budget (will be truncated, see truncate_to_tokens below)
~2.1% of human translations exceed the 3072-token generation budget (a model matching that length would get cut off)


## Prompt template

- Identical structure across models (system + user turn, folded for Gemma)
- `source_lang` is filled in per paragraph ('German' / 'Russian' / 'Chinese')
- No length instruction (see title cell for why)

In [8]:
SYSTEM_PROMPT = (
    'You are a professional literary translator contributing English '
    'translations to an anthology of world literature.'
)

USER_TEMPLATE = (
    'Translate the following {source_lang} passage into English.\n\n'
    'Text: {source_text}\n\n'
    'Preserve the meaning, tone, and structure of the original as closely as '
    'possible. Output only the English translation, with no title, no '
    'preamble, and no commentary.'
)


def build_messages(source_lang, source_text):
    user_content = USER_TEMPLATE.format(source_lang=source_lang, source_text=source_text)

    # Gemma's chat template has no 'system' role (only 'user'/'model'), so the
    # system prompt is folded into the user turn instead.
    if MODEL_KEY == 'gemma':
        return [
            {'role': 'user', 'content': f'{SYSTEM_PROMPT}\n\n{user_content}'},
        ]

    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]

## Generation (vLLM)

Passing `n=5` in `SamplingParams` and not five separate requests

In [9]:
# Check GPU availability
!nvidia-smi

Fri Aug 14 14:44:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [10]:
# Check installed versions of torch and vllm
!pip show torch vllm | grep -E "Name|Version"
!find / -name "libcudart.so*"

Name: torch
Version: 2.13.0
Name: vllm
Version: 0.27.1
find: ‘/proc/90/task/90/net’: Invalid argument
find: ‘/proc/90/net’: Invalid argument
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12
/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libcudart.so.13
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12.8.90
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so


In [14]:
# Load the CUDA runtime library with global symbol visibility so that vLLM can find it
import ctypes
ctypes.CDLL(
    '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libcudart.so.13',
    mode=ctypes.RTLD_GLOBAL,
)

import sys

# Jupyter/Colab's stdout and stderr (ipykernel.iostream.OutStream) do not
# implement fileno(), but vLLM calls sys.stdout.fileno() to briefly silence
# NCCL's native logging when it sets up the process group. Fake the real
# OS-level fd numbers (1 = stdout, 2 = stderr) so that call succeeds.
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

import glob, os

# Keep vLLM's engine in this process instead of a spawned subprocess. In a
# spawned child, ipykernel re-installs its own OutStream as sys.stdout, so
# the fileno() patch above (applied only in this process) would not carry
# over and the same crash would recur inside the child.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

nvidia_lib_dirs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/*/lib")
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print(nvidia_lib_dirs)

import sys, types
sys.modules['torchaudio'] = types.ModuleType('torchaudio')


from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

['/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cufile/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nvshmem/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nvtx/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cupti/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cusparselt/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cccl/lib', '/usr/local/lib/python3.12/dist-packages/nvidia/cublas/li

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def truncate_to_tokens(text, max_tokens):
    """Truncate text to at most max_tokens tokens under this model's tokenizer."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text
    return tokenizer.decode(ids[:max_tokens])


# Apply each model's own chat template and hand plain text to vLLM
text_prompts = [
    tokenizer.apply_chat_template(
        build_messages(item['source_lang'], truncate_to_tokens(item['source_text'], SRC_MAX_TOKENS)),
        tokenize=False,
        add_generation_prompt=True,
    )
    for item in pooled
]

print(f'Prepared {len(text_prompts)} prompts')
print(f'Source token budget: {SRC_MAX_TOKENS} (model context {MAX_MODEL_LEN})')

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Prepared 200 prompts
Source token budget: 3072 (model context 8192)


In [16]:
# Install the exact torch/torchvision/torchaudio versions vLLM 0.27.1 declares as dependencies
!pip install -q torch==2.13.0 torchvision==0.28.0



In [17]:
# Clear vLLM's model registry cache to ensure that the latest model classes are used
from vllm.model_executor.models import registry
registry._try_inspect_model_cls.cache_clear()

In [18]:
# Initialize the vLLM model: use bfloat16 for speed and memory efficiency, and
# limit GPU memory utilization to 90%.
# max_model_len is much larger than in the sibling notebooks (2048-4096)
# because of the long-tail paragraph lengths in this pool (see Length
# distribution above). If this OOMs on an L4, lower gpu_memory_utilization or
# switch to an A100.
llm = LLM(
    model=MODEL_ID,
    dtype='bfloat16',
    gpu_memory_utilization=0.90,
    max_model_len=MAX_MODEL_LEN,
    seed=SEED,
)

# Set up the sampling parameters for text generation
sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    repetition_penalty=REPETITION_PENALTY,
    seed=SEED,
)

# Generate translations for the pooled paragraphs using vLLM
start = time.time()
outputs = llm.generate(text_prompts, sampling_params)

print(f'Generation took {time.time() - start:.1f}s')

INFO 08-14 14:57:44 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'google/gemma-2-9b-it'}
WARNING 08-14 14:57:45 [arg_utils.py:1678] The global random seed is set to 42. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 08-14 14:58:01 [model.py:645] Resolved architecture: Gemma2ForCausalLM
INFO 08-14 14:58:01 [model.py:1883] Using max model len 8192
INFO 08-14 14:58:01 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-14 14:58:01 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

INFO 08-14 14:58:12 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='google/gemma-2-9b-it', speculative_config=None, tokenizer='google/gemma-2-9b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detaile

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

INFO 08-14 14:59:17 [weight_utils.py:530] Time spent downloading weights for google/gemma-2-9b-it: 58.762017 seconds
INFO 08-14 14:59:18 [weight_utils.py:867] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 17.21 GiB. Available RAM: 76.31 GiB.
INFO 08-14 14:59:18 [weight_utils.py:890] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


...

Rendering prompts:   0%|          | 0/200 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 08-14 15:02:03 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
WARNING 08-14 15:02:03 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _gumbel_sample_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 1000/1000 [06:18<00:00,  2.64it/s, est. speed input: 1528.90 toks/s, output: 1252.81 toks/s]

Generation took 379.5s


In [19]:
results = []

# Collect the generated translations and their metadata into a list of dictionaries
for item, output in zip(pooled, outputs):
    for k, candidate in enumerate(output.outputs):
        translation = candidate.text.strip()
        results.append({
            'prompt_id': item['prompt_id'],
            'book_id': item['book_id'],
            'source_lang': item['source_lang'],
            'paragraph_index': item['paragraph_index'],
            'model': MODEL_ID,
            'sample_idx': k,
            'translation': translation,
            'n_sentences': n_sentences(translation),
            'finish_reason': candidate.finish_reason,
            'n_tokens': len(candidate.token_ids),
            'source_text': item['source_text'],
            'gt_translation': item['gt_translation'],
            'human_translations': item['human_translations'],
        })

print(f'Collected {len(results)} translations')

Collected 1000 translations


In [20]:
# Language drift check: flag translations that still contain leftover
# non-Latin script. Unlike the other three (English-only) tasks, this also
# needs to cover Cyrillic (Russian source), not just CJK (Chinese source).
NON_LATIN = re.compile(r'[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af\u0400-\u04ff]')

for record in results:
    hits = NON_LATIN.findall(record['translation'])
    record['non_latin_chars'] = len(hits)
    record['language_ok'] = len(hits) == 0

flagged = [r for r in results if not r['language_ok']]

print(f'Translations with leftover non-Latin script: {len(flagged)} of {len(results)}')

for record in flagged:
    share = record['non_latin_chars'] / len(record['translation'])
    print(f"  {record['prompt_id']}/{record['sample_idx']}  ({record['source_lang']})  {share:.0%} non-Latin")


Translations with leftover non-Latin script: 64 of 1000
  par3_0004/1  (Chinese)  0% non-Latin
  par3_0020/2  (Chinese)  0% non-Latin
  par3_0020/4  (Chinese)  0% non-Latin
  par3_0022/2  (Chinese)  1% non-Latin
  par3_0022/3  (Chinese)  1% non-Latin
  par3_0025/2  (Chinese)  0% non-Latin
  par3_0046/1  (Chinese)  0% non-Latin
  par3_0046/3  (Chinese)  0% non-Latin
  par3_0046/4  (Chinese)  0% non-Latin
  par3_0052/2  (Russian)  1% non-Latin
  par3_0055/0  (Chinese)  1% non-Latin
  par3_0055/4  (Chinese)  1% non-Latin
  par3_0058/1  (Chinese)  1% non-Latin
  par3_0063/1  (Chinese)  1% non-Latin
  par3_0063/2  (Chinese)  1% non-Latin
  par3_0063/4  (Chinese)  1% non-Latin
  par3_0065/2  (Chinese)  1% non-Latin
  par3_0069/3  (Chinese)  0% non-Latin
  par3_0070/2  (Russian)  0% non-Latin
  par3_0083/2  (Chinese)  1% non-Latin
  par3_0083/3  (Chinese)  1% non-Latin
  par3_0084/0  (Chinese)  0% non-Latin
  par3_0084/1  (Chinese)  0% non-Latin
  par3_0084/2  (Chinese)  0% non-Latin
  par3_0

## Save results

In [21]:
out_path = OUT_DIR / f'par3_{MODEL_KEY}_n{N_SAMPLES}.jsonl'

# Save the generated translations to a JSONL file
with open(out_path, 'w') as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


# Store the configuration (own filename, so it does not collide with the
# WritingPrompts/SummEval/Multi-News config_{MODEL_KEY}.json written to the same OUT_DIR)
config = {
    'model_id': MODEL_ID,
    'dataset': 'Par3 (Karpinska et al., 2022), pooled German/Russian/Chinese source books',
    'pool_languages': list(POOL_LANGUAGES.values()),
    'min_sentences': MIN_SENTENCES,
    'min_translators': MIN_TRANSLATORS,
    'n_paragraphs_pooled': len(pooled_all),
    'n_paragraphs': len(pooled),
    'sample_quota_by_language': quotas,
    'n_samples': N_SAMPLES,
    'seed': SEED,
    'temperature': TEMPERATURE,
    'top_p': TOP_P,
    'max_new_tokens': MAX_NEW_TOKENS,
    'max_model_len': MAX_MODEL_LEN,
    'src_max_tokens': SRC_MAX_TOKENS,
    'repetition_penalty': REPETITION_PENALTY,
    'system_prompt': SYSTEM_PROMPT,
    'user_template': USER_TEMPLATE,
    'note': (
        'Paragraphs pooled across German/Russian/Chinese source books, filtered to '
        'indices where >= min_translators different human translators simultaneously '
        'reach >= min_sentences sentences (see data_description/descriptive_analysis_par3.ipynb), '
        'then stratified-sampled down to n_paragraphs, split as evenly as possible across '
        'the three source languages (sample_quota_by_language) so the much larger Russian pool '
        'does not swamp German. '
        'French is structurally excluded (no French book has >= min_translators translators).'
    ),
}

with open(OUT_DIR / f'par3_config_{MODEL_KEY}.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)


print('Saved:', out_path)

Saved: /content/drive/MyDrive/Masterarbeit/generations/par3_gemma_n5.jsonl


## Sanity checks

- `finish_reason == 'length'`: translation was cut off mid-sentence
- Length ratio vs. the human references: a translation wildly shorter or
  longer than every human translator suggests paraphrase-level drift rather
  than translation
- Non-Latin leftovers and repeated openings

In [22]:
import collections
import statistics as st

# Compute and print summary statistics for the generated translations
lengths = [len(record['translation'].split()) for record in results]
truncated = sum(1 for record in results if record['finish_reason'] == 'length')

print(f'Translations:  {len(results)}')
print(f'Median words:  {st.median(lengths):.0f}')
print(f'Mean words:    {st.mean(lengths):.0f} (SD {st.stdev(lengths):.0f})')
print(f'Min / max:     {min(lengths)} / {max(lengths)}')
print(f'Truncated:     {truncated} ({100 * truncated / len(results):.1f}%)')

print()
print('Truncation rate by source language:')
by_lang = collections.defaultdict(list)
for r in results:
    by_lang[r['source_lang']].append(r['finish_reason'] == 'length')
for lang, flags in sorted(by_lang.items()):
    print(f'  {lang}: {sum(flags)}/{len(flags)} ({100 * sum(flags) / len(flags):.1f}%)')

Translations:  1000
Median words:  286
Mean words:    369 (SD 284)
Min / max:     55 / 2154
Truncated:     0 (0.0%)

Truncation rate by source language:
  Chinese: 0/335 (0.0%)
  German: 0/335 (0.0%)
  Russian: 0/330 (0.0%)


In [23]:
# Share of generations that still reach MIN_SENTENCES -> relevant later since D_narr/D_disc need this

sentence_counts = [record['n_sentences'] for record in results]
below_floor = sum(1 for n in sentence_counts if n < MIN_SENTENCES)

print(f'Median sentences: {st.median(sentence_counts):.0f}')
print(f'Mean sentences:   {st.mean(sentence_counts):.1f} (SD {st.stdev(sentence_counts):.1f})')
print(f'Below {MIN_SENTENCES}-sentence floor: {below_floor} ({100 * below_floor / len(results):.1f}%)')

# Per-prompt: how many of the 5 samples survive the floor (need >= 2 for any pairwise diversity metric)
by_prompt = collections.defaultdict(list)
for r in results:
    by_prompt[r['prompt_id']].append(r['n_sentences'] >= MIN_SENTENCES)

survivors = [sum(v) for v in by_prompt.values()]
under_2 = sum(1 for s in survivors if s < 2)
print(f'Prompts with < 2 surviving generations (diversity metrics undefined): {under_2} of {len(survivors)}')

Median sentences: 17
Mean sentences:   20.8 (SD 15.2)
Below 10-sentence floor: 9 (0.9%)
Prompts with < 2 surviving generations (diversity metrics undefined): 0 of 200


In [24]:
# Length ratio vs. human references by source language

ratios_by_lang = collections.defaultdict(list)
for r in results:
    human_lens = [len(t.split()) for t in r['human_translations'].values()]
    ratio = len(r['translation'].split()) / np.mean(human_lens)
    ratios_by_lang[r['source_lang']].append(ratio)

print('Generated / human-reference word-count ratio (1.0 = same length as the average human translation):')
for lang, ratios in sorted(ratios_by_lang.items()):
    ratios = np.array(ratios)
    print(f'  {lang}: median={np.median(ratios):.2f} mean={ratios.mean():.2f} '
          f'p5={np.percentile(ratios, 5):.2f} p95={np.percentile(ratios, 95):.2f}')

Generated / human-reference word-count ratio (1.0 = same length as the average human translation):
  Chinese: median=0.98 mean=0.98 p5=0.83 p95=1.17
  German: median=0.90 mean=0.89 p5=0.74 p95=0.99
  Russian: median=0.92 mean=0.92 p5=0.85 p95=1.00


In [25]:
# Look for repeated openings

openings = collections.Counter(' '.join(r['translation'].split()[:3]) for r in results)

print('Most frequent openings:')
for opening, count in openings.most_common(5):
    print(f'  {count:3d}x  {opening}')

print()
print('--- Sample translation ---')
print(f"Source ({results[0]['source_lang']}, {results[0]['book_id']}): {results[0]['source_text'][:200]}")
print()
print(f"Generated: {results[0]['translation'][:400]}")

Most frequent openings:
   10x  One day, the
    7x  It is said
    5x  Even if Jade
    5x  K. was interrupted
    5x  The Monkey King

--- Sample translation ---
Source (Russian, notes_from_underground_ru): Впрочем, что ж я? — все это делают; болезнями-то и тщеславятся, а я, пожалуй, и больше всех. Не будем спорить; мое возражение нелепо. Но все-таки я крепко убежден, что не только очень много сознания, 

Generated: But what am I?  Everyone does this; they brag about their illnesses. And perhaps I do it more than anyone else. Let's not argue; my objection is foolish. But still, I am firmly convinced that not only is a great deal of consciousness an illness, but even all consciousness is an illness. I stand by that. Let us leave that for a moment. Tell me this: why was it that, as if on purpose, in precisely t
